# Lab 6: Hypothesis Testing
## Statistical Computing Lab

**Notebook objective:** Apply hypothesis-testing procedures to four practical scenarios using Python.

**Significance level:** \(\alpha = 0.05\) for all hypothesis tests.

### Problems covered
1. One-sample *t*-test — restaurant total bill versus $18.00.
2. Independent two-sample *t*-test — lunch versus dinner tips, preceded by Levene’s variance test.
3. Paired *t*-test — pulse at 1 minute versus 15 minutes using the `exercise` dataset.
4. Chi-square goodness-of-fit test — Titanic passenger-class proportions versus a 25% / 25% / 50% baseline.

> **Reproducibility note:** The notebook loads the datasets with Seaborn. The first run may require an internet connection if the datasets are not already cached. Problem 3 uses the dataset specified in the assignment request: `sns.load_dataset("exercise")`.


## Setup: Libraries and decision rule

We use `pandas` for data inspection, `seaborn` for the teaching datasets, `scipy.stats` for the statistical tests, and `matplotlib` for optional visual checks.

For each test, compare the p-value with \(\alpha = 0.05\):
- If **p-value < 0.05**, reject the null hypothesis.
- If **p-value ≥ 0.05**, fail to reject the null hypothesis.

A statistically significant result does not, by itself, establish practical importance or causation.


In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display

ALPHA = 0.05
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

print("Seaborn version:", sns.__version__)
print(f"Significance level (alpha): {ALPHA}")

Seaborn version: 0.13.2
Significance level (alpha): 0.05


---

## Problem 1 — One-Sample *t*-Test

### Scenario
The claimed population mean total bill is **$18.00**. We test whether the restaurant’s mean total bill differs from this benchmark.

### Hypotheses
- **Null hypothesis (H₀):** \(\mu = 18\)
- **Alternative hypothesis (H₁):** \(\mu \ne 18\)

This is a two-sided one-sample *t*-test. The sample consists of the `total_bill` observations in Seaborn’s `tips` dataset.


In [2]:
# Load and inspect the tips dataset
tips = sns.load_dataset("tips")
display(tips.head())
print(f"Rows in tips dataset: {len(tips)}")
print(f"Missing values in total_bill: {tips['total_bill'].isna().sum()}")

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


Rows in tips dataset: 244
Missing values in total_bill: 0


In [3]:
# Remove missing total_bill observations, then run the one-sample t-test
bills = tips["total_bill"].dropna()
claimed_mean = 18.00

t_stat_1, p_value_1 = stats.ttest_1samp(bills, popmean=claimed_mean)
sample_mean_1 = bills.mean()

print(f"Claimed mean:       ${claimed_mean:.2f}")
print(f"Observed sample mean: ${sample_mean_1:.2f}")
print(f"Sample size:        {len(bills)}")
print(f"t-statistic:        {t_stat_1:.4f}")
print(f"p-value:            {p_value_1:.6g}")

if p_value_1 < ALPHA:
    print(f"Decision: Reject H0 at alpha = {ALPHA:.2f}.")
    print("Conclusion: The mean total bill differs significantly from $18.00.")
else:
    print(f"Decision: Fail to reject H0 at alpha = {ALPHA:.2f}.")
    print("Conclusion: There is insufficient evidence that the mean total bill differs from $18.00.")

Claimed mean:       $18.00
Observed sample mean: $19.79
Sample size:        244
t-statistic:        3.1337
p-value:            0.00193837
Decision: Reject H0 at alpha = 0.05.
Conclusion: The mean total bill differs significantly from $18.00.


### Interpretation
Use the printed sample mean, *t*-statistic, and p-value to report the result. The decision rule is based on the two-sided alternative: evidence against H₀ is sufficient when the p-value is below 0.05.

**Assumptions to consider:** observations are independent, the response is quantitative, and the population distribution is approximately normal or the sample is large enough for the *t*-procedure to be robust.


---

## Problem 2 — Independent Two-Sample *t*-Test (Lunch vs Dinner)

### Scenario
We test whether average tip amounts differ between lunch and dinner services.

### Hypotheses
- **Null hypothesis (H₀):** \(\mu_{lunch} = \mu_{dinner}\)
- **Alternative hypothesis (H₁):** \(\mu_{lunch} \ne \mu_{dinner}\)

First, Levene’s test checks whether the group variances can reasonably be treated as equal. We then set `equal_var` according to that test, as instructed in the lab.


In [4]:
# Select non-missing tips for Lunch and Dinner
tips_by_time = tips.loc[tips["time"].isin(["Lunch", "Dinner"]), ["time", "tip"]].dropna()
lunch_tips = tips_by_time.loc[tips_by_time["time"] == "Lunch", "tip"]
dinner_tips = tips_by_time.loc[tips_by_time["time"] == "Dinner", "tip"]

print(f"Lunch sample size:  {len(lunch_tips)}")
print(f"Dinner sample size: {len(dinner_tips)}")
print(f"Lunch mean tip:     ${lunch_tips.mean():.3f}")
print(f"Dinner mean tip:    ${dinner_tips.mean():.3f}")

levene_stat, levene_p = stats.levene(lunch_tips, dinner_tips, center="median")
equal_variances = levene_p >= ALPHA

print("\nLevene's test for equality of variances")
print(f"Statistic: {levene_stat:.4f}")
print(f"p-value:   {levene_p:.6g}")
print("Variance decision:", "Treat variances as equal" if equal_variances else "Evidence of unequal variances")

Lunch sample size:  68
Dinner sample size: 176
Lunch mean tip:     $2.728
Dinner mean tip:    $3.103

Levene's test for equality of variances
Statistic: 0.8810
p-value:   0.348868
Variance decision: Treat variances as equal


In [5]:
# Independent two-sample t-test; equal_var is selected from Levene's test
t_stat_2, p_value_2 = stats.ttest_ind(lunch_tips, dinner_tips, equal_var=equal_variances)
mean_difference_2 = lunch_tips.mean() - dinner_tips.mean()

print(f"Test used: {'pooled-variance Student t-test' if equal_variances else 'Welch two-sample t-test'}")
print(f"Mean difference (Lunch - Dinner): ${mean_difference_2:.3f}")
print(f"t-statistic: {t_stat_2:.4f}")
print(f"p-value:     {p_value_2:.6g}")

if p_value_2 < ALPHA:
    print(f"Decision: Reject H0 at alpha = {ALPHA:.2f}.")
    print("Conclusion: Mean tip amounts differ significantly between lunch and dinner in this sample.")
else:
    print(f"Decision: Fail to reject H0 at alpha = {ALPHA:.2f}.")
    print("Conclusion: There is insufficient evidence of a difference in mean tip amounts between lunch and dinner.")

Test used: pooled-variance Student t-test
Mean difference (Lunch - Dinner): $-0.375
t-statistic: -1.9063
p-value:     0.0578015
Decision: Fail to reject H0 at alpha = 0.05.
Conclusion: There is insufficient evidence of a difference in mean tip amounts between lunch and dinner.


### Interpretation
Levene’s test determines which independent-samples test is used: if its p-value is at least 0.05, the analysis proceeds with `equal_var=True`; otherwise, it uses `equal_var=False` (Welch’s test).

The second p-value addresses whether the mean tip differs by service time. This observational comparison can identify an association, but it does not prove that service time causes any difference.


---

## Problem 3 — Paired *t*-Test (Pulse at 1 Minute vs 15 Minutes)

### Scenario
The exercise study records pulse measurements for participants at different time points. We test whether pulse changes between **1 minute** and **15 minutes** for matched observations.

### Hypotheses
Define each paired difference as \(d = pulse_{1\ min} - pulse_{15\ min}\).
- **Null hypothesis (H₀):** \(\mu_d = 0\) (no average change)
- **Alternative hypothesis (H₁):** \(\mu_d \ne 0\) (an average change)

**Dataset specified for this problem:**
```python
import seaborn as sns
exercise = sns.load_dataset("exercise")
```

The dataset has repeated records organized by participant-related study factors, including `id`, `time`, and `pulse`. We match the 1-minute and 15-minute observations using the available identifiers and study conditions so each difference compares like with like.


In [6]:
# Load the exact dataset requested
exercise = sns.load_dataset("exercise")
display(exercise.head())
print("Dataset shape:", exercise.shape)
print("\nColumns:", exercise.columns.tolist())
print("\nTime values:", exercise["time"].dropna().unique())
print("\nMissing values in relevant columns:")
display(exercise[["id", "diet", "kind", "time", "pulse"]].isna().sum().to_frame("missing_count"))

,Unnamed: 0,id,diet,pulse,time,kind
0,0,1,low fat,85,1 min,rest
1,1,1,low fat,85,15 min,rest
2,2,1,low fat,88,30 min,rest
3,3,2,low fat,90,1 min,rest
4,4,2,low fat,92,15 min,rest


Dataset shape: (90, 6)

Columns: ['Unnamed: 0', 'id', 'diet', 'pulse', 'time', 'kind']

Time values: ['1 min', '15 min', '30 min']
Categories (3, str): ['1 min', '15 min', '30 min']

Missing values in relevant columns:


,missing_count
id,0
diet,0
kind,0
time,0
pulse,0


In [7]:
# Match repeated pulse measurements at 1 min and 15 min.
# Include id, diet, and kind as matching keys to compare the same study condition.
exercise_pair_source = exercise.loc[
    exercise["time"].isin(["1 min", "15 min"]),
    ["id", "diet", "kind", "time", "pulse"]
].dropna(subset=["id", "diet", "kind", "time", "pulse"])

pulse_wide = exercise_pair_source.pivot_table(
    index=["id", "diet", "kind"],
    columns="time",
    values="pulse",
    aggfunc="mean"
).dropna(subset=["1 min", "15 min"])

pulse_1min = pulse_wide["1 min"]
pulse_15min = pulse_wide["15 min"]
pulse_diff = pulse_1min - pulse_15min

print(f"Matched pairs: {len(pulse_wide)}")
print(f"Mean pulse at 1 min:  {pulse_1min.mean():.3f}")
print(f"Mean pulse at 15 min: {pulse_15min.mean():.3f}")
print(f"Mean paired difference (1 min - 15 min): {pulse_diff.mean():.3f} beats/min")

Matched pairs: 30
Mean pulse at 1 min:  93.133
Mean pulse at 15 min: 101.533
Mean paired difference (1 min - 15 min): -8.400 beats/min


In [8]:
# Paired t-test on matched observations
t_stat_3, p_value_3 = stats.ttest_rel(pulse_1min, pulse_15min)

print(f"t-statistic: {t_stat_3:.4f}")
print(f"p-value:     {p_value_3:.6g}")

if p_value_3 < ALPHA:
    print(f"Decision: Reject H0 at alpha = {ALPHA:.2f}.")
    print("Conclusion: Mean pulse changed significantly between 1 minute and 15 minutes.")
else:
    print(f"Decision: Fail to reject H0 at alpha = {ALPHA:.2f}.")
    print("Conclusion: There is insufficient evidence of a mean pulse change between 1 minute and 15 minutes.")

t-statistic: -3.7271
p-value:     0.000834885
Decision: Reject H0 at alpha = 0.05.
Conclusion: Mean pulse changed significantly between 1 minute and 15 minutes.


### Interpretation and data-handling note
The paired test is performed only on matched 1-minute/15-minute records, and the reported average difference is **1 minute minus 15 minutes**. A positive mean difference indicates that pulse was higher at 1 minute on average; a negative value indicates it was higher at 15 minutes.

The `exercise` dataset’s repeated measurements are matched using `id`, `diet`, and `kind`. If the dataset structure changes in a future Seaborn release, inspect the printed column and time values before interpreting the result.


---

## Problem 4 — Chi-Square Goodness-of-Fit Test (Titanic Passenger Class)

### Scenario
A historical baseline proposes passenger-class proportions of **25% First Class, 25% Second Class, and 50% Third Class**. We compare the observed Titanic passenger-class counts with these expected proportions.

### Hypotheses
- **Null hypothesis (H₀):** Passenger-class proportions are 0.25, 0.25, and 0.50.
- **Alternative hypothesis (H₁):** The observed class distribution does not follow those proportions.

The test uses observed counts, not percentages. Expected counts are calculated as total count multiplied by each expected proportion.


In [9]:
# Load Titanic dataset and inspect class categories
titanic = sns.load_dataset("titanic")
display(titanic.head())
print("Dataset shape:", titanic.shape)
print("\nClass values:", titanic["class"].value_counts(dropna=False))

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


Dataset shape: (891, 15)

Class values: class
Third     491
First     216
Second    184
Name: count, dtype: int64


In [10]:
# Count observed passenger classes; exclude missing class labels
class_order = ["First", "Second", "Third"]
observed_series = titanic["class"].dropna().value_counts().reindex(class_order, fill_value=0)
observed = observed_series.to_numpy(dtype=int)
proportions = np.array([0.25, 0.25, 0.50], dtype=float)
total_passengers = observed.sum()
expected = total_passengers * proportions

count_table = pd.DataFrame({
    "Passenger class": class_order,
    "Observed count": observed,
    "Expected proportion": proportions,
    "Expected count": expected
})
display(count_table)
print(f"Total passengers included: {total_passengers}")
print("Expected-count array:", expected)

,Passenger class,Observed count,Expected proportion,Expected count
0,First,216,0.25,222.75
1,Second,184,0.25,222.75
2,Third,491,0.50,445.50


Total passengers included: 891
Expected-count array: [222.75 222.75 445.5 ]


In [11]:
# Chi-square goodness-of-fit test
chi2_stat_4, p_value_4 = stats.chisquare(f_obs=observed, f_exp=expected)

print(f"Chi-square statistic: {chi2_stat_4:.4f}")
print(f"Degrees of freedom:   {len(observed) - 1}")
print(f"p-value:              {p_value_4:.6g}")

if p_value_4 < ALPHA:
    print(f"Decision: Reject H0 at alpha = {ALPHA:.2f}.")
    print("Conclusion: The observed passenger-class distribution differs significantly from the 25% / 25% / 50% baseline.")
else:
    print(f"Decision: Fail to reject H0 at alpha = {ALPHA:.2f}.")
    print("Conclusion: There is insufficient evidence that the observed distribution differs from the 25% / 25% / 50% baseline.")

Chi-square statistic: 11.5926
Degrees of freedom:   2
p-value:              0.00303879
Decision: Reject H0 at alpha = 0.05.
Conclusion: The observed passenger-class distribution differs significantly from the 25% / 25% / 50% baseline.


### Interpretation
The chi-square statistic measures the discrepancy between observed and expected counts. A p-value below 0.05 indicates that the observed class counts are inconsistent with the stated baseline proportions.

**Scope note:** This analysis follows the assignment’s stated baseline and uses non-missing `class` labels in the Seaborn Titanic dataset. The conclusion concerns this dataset and the specified proportions; it does not independently validate the historical report.


---

## Final Summary

The table below collects the test statistics and p-values calculated above. The final decision is generated from the same 5% significance rule used throughout the notebook.


In [12]:
summary = pd.DataFrame([
    {
        "Problem": "1. One-sample t-test",
        "Test statistic": t_stat_1,
        "p-value": p_value_1,
        "Decision at alpha=0.05": "Reject H0" if p_value_1 < ALPHA else "Fail to reject H0"
    },
    {
        "Problem": "2. Independent two-sample t-test",
        "Test statistic": t_stat_2,
        "p-value": p_value_2,
        "Decision at alpha=0.05": "Reject H0" if p_value_2 < ALPHA else "Fail to reject H0"
    },
    {
        "Problem": "3. Paired t-test",
        "Test statistic": t_stat_3,
        "p-value": p_value_3,
        "Decision at alpha=0.05": "Reject H0" if p_value_3 < ALPHA else "Fail to reject H0"
    },
    {
        "Problem": "4. Chi-square goodness-of-fit",
        "Test statistic": chi2_stat_4,
        "p-value": p_value_4,
        "Decision at alpha=0.05": "Reject H0" if p_value_4 < ALPHA else "Fail to reject H0"
    }
])
display(summary.style.format({"Test statistic": "{:.4f}", "p-value": "{:.6g}"}))

,Problem,Test statistic,p-value,Decision at alpha=0.05
0,1. One-sample t-test,3.1337,0.00193837,Reject H0
1,2. Independent two-sample t-test,-1.9063,0.0578015,Fail to reject H0
2,3. Paired t-test,-3.7271,0.000834885,Reject H0
3,4. Chi-square goodness-of-fit,11.5926,0.00303879,Reject H0
